# Which supermarket is cheaper? Comparing by product type

`compare.ipynb` compares chains **by search term**, so the noise each search returns ("leche" → chocolate con leche,
leche corporal…) ends up in the averages. This notebook uses `data/products_normalized.csv`, where every product has
a **product type** (e.g. *leche semidesnatada*) or `no_relevante`, and compares like with like.

All prices come from `src/analysis/prices.py`, the same code the dashboard uses. Prices are € per kg, litre, unit
or wash, depending on the type. Rows flagged as outliers, without a price per unit, or in a different unit than
their type are left out (`prices.usable`).

1. Coverage · 2. Price by type · 3. Basket index · 4. Are the differences significant? · 5. Store brand vs brand ·
6. Closest equivalents · 7. How good is the classifier?

In [ ]:
import sys
from itertools import combinations
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pipeline").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon

from src.analysis import prices
from src.pipeline import config

df = prices.load_normalized()
usable = prices.usable(df)
CHAINS = sorted(usable["supermarket"].unique())
labels = (df.dropna(subset=["product_type"]).drop_duplicates("product_type")
            .set_index("product_type")["product_type_label"])
print(f"{len(df)} products, {len(usable)} usable for comparison ({len(usable) / len(df):.0%})")
df.groupby("supermarket").apply(lambda g: pd.Series({
    "products": len(g),
    "no_relevante": (g["product_type"] == config.NO_RELEVANTE).mean(),
    "usable": prices.usable(g).shape[0],
}), include_groups=False).round(2)

## 1. Coverage

How many usable products each chain has per type. A median built on one or two products is fragile, so
read the later tables with this one in mind.

In [ ]:
coverage = usable.pivot_table(index=["searched_item", "product_type"], columns="supermarket",
                              values="price_per_unit", aggfunc="size", fill_value=0)
coverage["chains"] = (coverage[CHAINS] > 0).sum(axis=1)
coverage

## 2. Price by type

Median € per unit for every type and chain. The colour is the price relative to the median of all chains for that
type (green = cheaper, red = more expensive), so types in different units can share one picture.

In [ ]:
types = list(coverage.index.get_level_values("product_type"))
median_table = prices.price_table(df, types, stat="median")[CHAINS]
min_table = prices.price_table(df, types, stat="min")[CHAINS]

relative = median_table.div(median_table.median(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(9, 0.28 * len(relative) + 1))
im = ax.imshow(np.log2(relative), cmap="RdYlGn_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(CHAINS)), CHAINS, rotation=30, ha="right")
ax.set_yticks(range(len(relative)), [labels.get(t, t) for t in relative.index], fontsize=8)
for (i, j), value in np.ndenumerate(median_table.values):
    if not np.isnan(value):
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, label="log2(price / median of chains)", shrink=0.5)
ax.set_title("Median € per unit by product type")
plt.tight_layout()
plt.show()

In [ ]:
# Cheapest product of each type per chain (€ per unit)
min_table.round(2)

## 3. Basket index

A fixed reference basket, like the OCU's yearly supermarket study: one everyday type per search term, with
quantities we chose (in the type's unit: litres, kg, units, washes). Each chain's basket is priced with the median
€ per unit of each type. To compare fairly, the index only uses the types **every** chain has; the cheapest chain = 100.

In [ ]:
BASKET = {
    "leche_semidesnatada": 6,       # 6 L
    "huevos_gallina": 12,           # 12 eggs
    "manzana_roja": 1,              # 1 kg
    "platano_canarias": 1,          # 1 kg
    "tomate_frito": 0.4,            # 400 g
    "arroz_redondo": 1,             # 1 kg
    "pasta_larga": 1,               # 1 kg
    "pan_molde": 0.5,               # 500 g
    "pollo_pechuga": 1,             # 1 kg
    "picada_vacuno": 0.5,           # 500 g
    "salmon_fresco": 0.5,           # 500 g
    "yogur_natural": 0.5,           # 4 × 125 g
    "queso_curado": 0.25,           # 250 g
    "agua_sin_gas": 6,              # 6 L
    "zumo_naranja": 1,              # 1 L
    "atun_aceite_oliva": 0.24,      # 3 × 80 g
    "garbanzo_cocido": 0.4,         # 400 g
    "aceite_oliva_virgen_extra": 1, # 1 L
    "detergente_liquido": 40,       # 40 washes
    "papel_higienico": 12,          # 12 rolls
}

basket_prices = prices.price_table(df, list(BASKET), stat="median")[CHAINS]
common = basket_prices.dropna().index
print(f"{len(common)} of {len(BASKET)} basket types are sold by all {len(CHAINS)} chains:", ", ".join(common))

costs = pd.Series({chain: prices.basket_cost(df, {t: BASKET[t] for t in common}, chain)[0] for chain in CHAINS})
index = pd.DataFrame({"basket €": costs.round(2), "index (cheapest = 100)": (100 * costs / costs.min()).round(1)})
index["missing from full basket"] = [len(prices.basket_cost(df, BASKET, c)[1]) for c in CHAINS]
index.sort_values("basket €")

In [ ]:
ax = index["index (cheapest = 100)"].sort_values().plot.barh(figsize=(6, 3), color="#4C78A8")
ax.axvline(100, color="grey", lw=0.8)
ax.set_xlim(left=90)
ax.set_title(f"Basket index ({len(common)} common types)")
ax.set_xlabel("cheapest chain = 100")
plt.tight_layout()
plt.show()

## 4. Are the differences significant?

- **Wilcoxon signed-rank test** for each pair of chains, paired on the types both sell (median € per unit).
  The question: is one chain systematically cheaper type by type?
- **Friedman test** over the types all chains share: are there differences between chains at all?
- With 21 pairs we correct the p-values with **Holm's method**, so "significant" isn't just luck from many tests.

In [ ]:
def holm(p_values):
    """Holm-Bonferroni adjusted p-values (same order as the input)."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adjusted[i] = min(running, 1.0)
    return adjusted

rows = []
for a, b in combinations(CHAINS, 2):
    pair = median_table[[a, b]].dropna()
    if len(pair) < 6:
        continue
    stat = wilcoxon(pair[a], pair[b])
    rows.append({"chain A": a, "chain B": b, "shared types": len(pair),
                 "median A/B": (pair[a] / pair[b]).median(), "p": stat.pvalue})
pairs = pd.DataFrame(rows)
pairs["p (Holm)"] = holm(pairs["p"])
pairs["A cheaper?"] = np.where(pairs["p (Holm)"] < 0.05,
                               np.where(pairs["median A/B"] < 1, "yes", "no, B is"), "not significant")
pairs.round(3).sort_values("p (Holm)")

In [ ]:
shared = median_table.dropna()
stat = friedmanchisquare(*[shared[c] for c in CHAINS])
print(f"Friedman test on {len(shared)} types sold by all chains: chi² = {stat.statistic:.1f}, p = {stat.pvalue:.4f}")
shared.rank(axis=1).mean().sort_values().rename("mean rank (1 = cheapest)").round(2)

## 5. Store brand vs manufacturer brand

For each type and chain, the median € per unit of the chain's own brand (Hacendado, Dia, Milsani, Auchan…)
divided by that of the manufacturer brands. Below 1 means the store brand is cheaper.

In [ ]:
by_brand = usable.pivot_table(index=["product_type", "supermarket"], columns="is_private_label",
                              values="price_per_unit", aggfunc="median").dropna()
by_brand.columns = ["brand", "store brand"]
by_brand["ratio"] = by_brand["store brand"] / by_brand["brand"]
summary = by_brand.groupby("supermarket")["ratio"].agg(["median", "count"]).rename(
    columns={"median": "store brand / brand (median)", "count": "types compared"})
summary.round(2).sort_values("store brand / brand (median)")

## 6. Closest equivalents across chains

`find_similar` searches by text similarity (the same vectors the classifier uses). For a few Mercadona products
it finds the closest product of the same type in every other chain and compares the price per unit.

In [ ]:
from src.pipeline.build import prepare
from src.pipeline.classify import decide, find_similar

p = prepare()
classification = decide(p["products"], p["hits"], p["taxonomy"], p["vectorizer"],
                        config.THRESHOLDS[p["vectorizer"].method])
ppu = df.set_index("product_key")["price_per_unit"]
examples = (usable[(usable["supermarket"] == "Mercadona") & usable["product_type"].isin(list(BASKET))]
            .sample(5, random_state=config.SEED))

rows = []
for item in examples.itertuples():
    for chain in config.CHAINS:
        if chain == "mercadona":
            continue
        match = find_similar(item.product_key, k=1, chain=chain, type_id=item.product_type,
                             products=p["products"], classification=classification, vectorizer=p["vectorizer"])
        if match.empty:
            continue
        m = match.iloc[0]
        rows.append({"Mercadona product": item.name, "€/unit": item.price_per_unit, "chain": chain,
                     "closest product": m["name"], "similarity": round(m["similarity"], 2),
                     "its €/unit": ppu.get(m["product_key"]),
                     "difference": ppu.get(m["product_key"]) / item.price_per_unit - 1})
pd.DataFrame(rows).round(2)

## 7. How good is the classifier?

Measured on hand-labelled products (`src/pipeline.ipynb`, labelling cell). Half of the labels tuned the thresholds,
the other half gives the numbers below. The 95% interval is wide because the test set is small (~50 products).

In [ ]:
from src.pipeline.evaluate import evaluate, load_labels

try:
    result = evaluate(p, load_labels())
    r = result["report"]
    print(f"{result['method']}: accuracy {r['accuracy']:.2f} (95% CI {r['accuracy_ci95'][0]:.2f}–{r['accuracy_ci95'][1]:.2f}), "
          f"macro F1 {r['macro_f1']:.2f}, relevant precision {r['relevant_precision']:.2f}, "
          f"relevant recall {r['relevant_recall']:.2f} (n = {r['n']})")
except ValueError as e:
    print("Not evaluated yet:", e)